# Figure L00: the six-frame ORF classes (schematic)

**Status:** draft, Setonix (George's review of #150, comment on the LOGAN ORF-space subsection).

## Provenance

- **Data:** none. The contig, ORFs and stop codons are illustrative coordinates hard-coded below, not a real locus.
- **Classes:** the Prodigal+ / Prodigal− classes of Table 2 and *Contigs, ORFs and Prodigal labels* (Methods): an ORF is Prodigal+ when a Prodigal gene on the same contig and strand ends at the same stop and passes the acceptance criteria; a Prodigal− ORF is classed by its largest-overlapping Prodigal gene as an antisense shadow (opposite strand), a frameshift shadow (same strand, other frame), a marginal overlap (1–29 bp), or intergenic (no overlap). ORFs are maximal stop to stop, so each begins after an in-frame stop and ends at the next one, unless it runs off a contig end.


In [ ]:
import msfig
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Patch
msfig.style()
P = msfig.PALETTE

L = 3000  # dummy contig, bp
H = 0.56   # arrow height
TRACK = {"+1": 6, "+2": 5, "+3": 4, "-1": 2, "-2": 1, "-3": 0}
DNA_Y = 3

CLS = {
    "complete": ("Prodigal+ complete gene", P["green"]),
    "partial": ("Prodigal+ partial gene (open at contig end)", "#7FCEB9"),
    "antisense": ("antisense shadow", P["vermillion"]),
    "frameshift": ("frameshift shadow", P["orange"]),
    "intergenic": ("intergenic", P["blue"]),
    "marginal": ("marginal overlap (<30 bp)", P["sky"]),
}
# (frame, start, end, class, open5, open3) -- illustrative coordinates only
ORFS = [
    ("+3", 0, 260, "partial", True, False),
    ("+1", 300, 1200, "complete", False, False),
    ("-2", 420, 1080, "antisense", False, False),
    ("+2", 690, 1310, "frameshift", False, False),
    ("-1", 1330, 1720, "intergenic", False, False),
    ("+1", 1780, 1921, "marginal", False, False),
    ("+3", 1900, 2560, "complete", False, False),
    ("+1", 2650, 3000, "partial", False, True),
]
# extra in-frame stops (vertical ticks), as Artemis draws them
STOPS = {"+1": [120, 297, 1620, 1777, 2290, 2647], "+2": [180, 440, 687, 1690, 2210, 2840],
         "+3": [1460, 1897, 2780], "-1": [260, 900, 1723, 2130, 2600],
         "-2": [150, 1083, 1500, 2010, 2470, 2920], "-3": [520, 1250, 1870, 2380]}

fig, ax = plt.subplots(figsize=(7.2, 2.9))
for f, y in TRACK.items():
    ax.axhspan(y - 0.42, y + 0.42, color="#F2F2F2", lw=0, zorder=0)
    ax.text(-40, y, f"frame {f.replace('-', '−')}", ha="right", va="center", fontsize=7)
    for s in STOPS[f]:
        ax.plot([s, s], [y - 0.42, y + 0.42], color="#222222", lw=0.8, zorder=1)

for f, a, b, c, o5, o3 in ORFS:
    y = TRACK[f]
    fwd = f.startswith("+")
    _, col = CLS[c]
    tip = 60
    if fwd:
        pts = ([(a, y - H/2), (b - tip, y - H/2), (b, y), (b - tip, y + H/2), (a, y + H/2)]
               if not o3 else [(a, y - H/2), (b, y - H/2), (b, y + H/2), (a, y + H/2)])
    else:
        pts = [(b, y - H/2), (a + tip, y - H/2), (a, y), (a + tip, y + H/2), (b, y + H/2)]
    ax.add_patch(Polygon(pts, closed=True, fc=col, ec="#222222", lw=0.6, zorder=3,
                         ls="--" if (o5 or o3) else "-"))
    # stop codon that ends the ORF (none at an open end)
    end = b if fwd else a
    if not (o3 and fwd):
        ax.plot([end, end], [y - 0.42, y + 0.42], color="#222222", lw=0.8, zorder=4)
    if o5:
        ax.text(a + 15, y, "5′ open", fontsize=6, ha="left", va="center", zorder=5)
    if o3:
        ax.text(b - 15, y, "3′ open", fontsize=6, ha="right", va="center", zorder=5)

# DNA axis
ax.plot([0, L], [DNA_Y, DNA_Y], color="#222222", lw=1.2)
for x in range(0, L + 1, 500):
    ax.plot([x, x], [DNA_Y - 0.12, DNA_Y + 0.12], color="#222222", lw=0.8)
    ax.text(x, DNA_Y - 0.22, f"{x:,}", ha="center", va="top", fontsize=6)
ax.text(-40, DNA_Y, "contig (bp)", ha="right", va="center", fontsize=7)
ax.text(L / 2, 6.62, "forward strand", ha="center", va="bottom", fontsize=7, color="#555555")
ax.text(L / 2, -0.62, "reverse strand", ha="center", va="top", fontsize=7, color="#555555")
for x in (0, L):
    ax.plot([x, x], [-0.45, 6.45], color="#555555", lw=0.6, ls=(0, (2, 2)), zorder=0)

handles = [Patch(fc=CLS[k][1], ec="#222222", lw=0.6, label=CLS[k][0]) for k in CLS]
handles.append(plt.Line2D([], [], color="#222222", lw=0.8, label="in-frame stop codon"))
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.04), ncol=4,
          fontsize=6.5, handlelength=1.6, columnspacing=1.2)
ax.set_xlim(-20, L + 20)
ax.set_ylim(-0.9, 6.95)
ax.axis("off")
msfig.save(fig, "figL00_orf_classes")


## Caption (draft)

**Figure L00. The classes of six-frame ORFs, on an illustrative contig.** Each row is one reading frame (three on the forward strand, three on the reverse), drawn as in Artemis: vertical ticks are in-frame stop codons and every ORF runs from one stop to the next. Green: ORFs that match a Prodigal gene (Prodigal+), complete or open at a contig end (dashed outline, light green; 5′ open on the left, 3′ open on the right). The other ORFs (Prodigal−) are classed by the Prodigal gene they overlap most: an antisense shadow on the opposite strand (vermillion), a frameshift shadow in another frame of the same strand (orange), a marginal overlap of fewer than 30 bp (sky blue), or no overlap at all, intergenic (blue). Coordinates are illustrative.
